### Create KAKEN work-funder/award linkages (funder-reported)

Resolves the per-project research outputs that KAKEN investigators self-report (captured as
`products_json` on `openalex.awards.kaken_projects_raw` by `scripts/local/kaken_to_s3.py`) to
OpenAlex works, and writes a junction table `openalex.awards.kaken_work_funders` analogous to
`nwo_work_funders` / `anr_work_funders`: one row per `(work_id, funder_id)` with the KAKEN
`award_ids` (= project ids).

**Why this exists:** KAKEN was ingested before OpenAlex had the funder-asserted
grant->publication pipelines (NWO, ANR, DataCite, EuropePMC), so its self-reported linkages
were never wired into `WorkAwards`. The KAKEN grant pages list every reported output under
"Research Products"; the scraper now captures each output's DOI, so we can resolve them here.

**Work resolution (DOI path):** KAKEN reports a clean bare DOI (`10.xxxx/yyyy`) for the bulk of
its journal-article outputs (~96% of journal articles on a sampled large grant). We normalize it
to the canonical `https://doi.org/` form and join `openalex.works.openalex_works.doi`. DOI-less
outputs (older / Japanese-language items, presentations, books) are skipped for now; a NAID and
title-based fuzzy fallback is a tracked follow-on.

**Award entities are NOT created here** — KAKEN awards already exist in `openalex_awards`
(provenance `kaken`, via `CreateKAKENAwards`). We only confirm the award exists and emit the
edge. The downstream `WorkAwards` notebook joins this junction to the existing entity.

Feeds: **WorkAwards** (new `kaken_work_funder_awards` leg, joins to `openalex_awards`).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kaken_work_funders
USING delta
AS
WITH products AS (
  SELECT
    pr.project_id,
    p.prod['doi']  AS doi_raw,
    p.prod['type'] AS product_type
  FROM openalex.awards.kaken_projects_raw pr
  LATERAL VIEW EXPLODE(from_json(pr.products_json, 'array<map<string,string>>')) p AS prod
  WHERE pr.products_json IS NOT NULL
    AND pr.products_json NOT IN ('', '[]', 'null')
    AND p.prod['doi'] IS NOT NULL
),
-- Normalize the bare KAKEN DOI ("10.xxxx/yyyy") to the canonical doi.org URL and resolve it
-- against OpenAlex works. NOTE: regex backslashes are DOUBLE-escaped ('\\.', '\\s') because the
-- job cluster runs in legacy escapedStringLiterals mode, where a single '\s' collapses to 's'
-- and silently shrinks the match (repo convention, see NWO/oxjobs #244).
doi_resolved AS (
  SELECT DISTINCT
    pr.project_id,
    w.id AS work_id
  FROM (
    SELECT
      project_id,
      CONCAT('https://doi.org/',
             regexp_extract(lower(trim(doi_raw)), '(10\\.[0-9]{2,}/[^\\s"<>]+)', 1)) AS doi_url
    FROM products
    WHERE lower(doi_raw) RLIKE '10\\.[0-9]'
  ) pr
  JOIN openalex.works.openalex_works w
    ON lower(w.doi) = pr.doi_url
  WHERE pr.doi_url <> 'https://doi.org/'
),
-- Confirm the KAKEN award entity exists (provenance 'kaken') and pick up its funder_id.
-- kaken_awards.funder_award_id == KAKEN project id (1:1).
with_award AS (
  SELECT
    r.work_id,
    a.funder_id,
    a.funder_award_id
  FROM doi_resolved r
  JOIN openalex.awards.kaken_awards a
    ON a.funder_award_id = r.project_id
  WHERE r.work_id IS NOT NULL
)
SELECT
  work_id,
  funder_id,
  ARRAY_DISTINCT(COLLECT_LIST(funder_award_id)) AS award_ids
FROM with_award
GROUP BY work_id, funder_id

### Sanity checks

In [ ]:
%sql
-- Row counts, key uniqueness, and award fill
SELECT
  COUNT(*)                                              AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id))      AS distinct_keys,
  COUNT(DISTINCT work_id)                              AS distinct_works,
  COUNT(DISTINCT funder_id)                            AS distinct_funders,
  SUM(SIZE(award_ids))                                 AS total_edges,
  COUNT(CASE WHEN SIZE(award_ids) > 1 THEN 1 END)      AS works_multi_award
FROM openalex.awards.kaken_work_funders

In [ ]:
%sql
-- Composite-key uniqueness (expect 0) and project-level coverage vs. the source population
WITH dup AS (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.kaken_work_funders
  GROUP BY work_id, funder_id HAVING n > 1
),
projects_with_products AS (
  SELECT COUNT(*) AS n
  FROM openalex.awards.kaken_projects_raw
  WHERE products_json IS NOT NULL AND products_json NOT IN ('', '[]', 'null')
),
resolved_projects AS (
  SELECT COUNT(DISTINCT a.funder_award_id) AS n
  FROM (
    SELECT EXPLODE(award_ids) AS award_id
    FROM openalex.awards.kaken_work_funders
  ) t
  JOIN openalex.awards.kaken_awards a ON a.funder_award_id = t.award_id
)
SELECT
  (SELECT COUNT(*) FROM dup)                                                AS duplicate_keys,
  (SELECT n FROM projects_with_products)                                    AS projects_with_products,
  (SELECT n FROM resolved_projects)                                         AS projects_with_a_linked_work,
  ROUND(100.0 * (SELECT n FROM resolved_projects)
              / NULLIF((SELECT n FROM projects_with_products), 0), 1)       AS pct_projects_covered

In [ ]:
%sql
-- Spot-check a few resolved edges
SELECT work_id, funder_id, award_ids
FROM openalex.awards.kaken_work_funders
ORDER BY SIZE(award_ids) DESC
LIMIT 10